In [1]:
# Install compatible dependencies
!pip install --upgrade numpy==1.26.4 pandas==2.2.2 scikit-learn peft wandb tqdm -q

In [ ]:
import os
# Restart the kernel to use the newly installed library versions
os.kill(os.getpid(), 9)

# SOMA — Experiment 1: Permuted MNIST

**Wakasa Labs · Nairobi, Kenya · March 2026**

This notebook runs the full SOMA system on 10 permuted MNIST tasks.
Designed to run on **Kaggle T4 GPU** (~15 min runtime).

**PASS criterion:** BT > -0.05 AND K < 10

In [9]:
# Clone SOMA repo if not exists and add to path
import os
import sys
import importlib

if not os.path.exists('soma_research'):
    !git clone https://github.com/LensenWakasa/SOMA-research.git soma_research

# Ensure we have the absolute path
repo_path = os.path.abspath('soma_research')

# Add to sys.path and move to front
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Force a path refresh for the current process
importlib.invalidate_caches()

# Verify import with full path checking
try:
    import soma
    print(f'Soma package found at: {soma.__file__}')
    from soma.core.necessity import SomaNecessity
    from soma.core.grow import SomaGrow
    from soma.core.learn import SomaLearn
    print('SOMA modules loaded successfully')
except ImportError as e:
    print(f'Import failed: {e}')
    print('Current sys.path:', sys.path)

Soma package found at: /content/soma_research/soma/__init__.py
SOMA modules loaded successfully


In [10]:
# GPU check
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    # Fixed attribute name from total_mem to total_memory
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [18]:
# Pull latest SOMA code with fixes (MERGE masking, warmup guard, BT tracking)
import subprocess
import sys
import os
import importlib

os.chdir('soma_research')
subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True)
os.chdir('..')

# Reload all SOMA modules to pick up latest code
modules_to_reload = [
    'soma.core.necessity',
    'soma.core.grow',
    'soma.core.learn',
    'soma.core.router',
    'soma.experiments.run_permuted_mnist'
]
for mod_name in modules_to_reload:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

# Run experiment
from soma.experiments.run_permuted_mnist import run_experiment
import argparse
import torch

args = argparse.Namespace(
    n_tasks=10, n_train=1000, n_test=200,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=42, no_rl=False, disable_n1=False, disable_n2=False, disable_n3=False,
)

result = run_experiment(args)

=== SOMA Experiment 1: Permuted MNIST ===
Tasks: 10, Train: 1000, Test: 200
Device: cuda, Seed: 42

Generating permuted MNIST tasks...
[Task 1/10]
  Action: SPAWN(cold)  K: 0->1  Acc: 0.815  BT: 0.0000
[Task 2/10]
  Action: SPAWN(cold)  K: 1->2  Acc: 0.835  BT: 0.0000
[Task 3/10]
  Action: SPAWN_NEW  K: 2->3  Acc: 0.875  BT: 0.0000
[Task 4/10]
  Action: SPAWN_NEW  K: 3->4  Acc: 0.895  BT: 0.0000
[Task 5/10]
  Action: SKIP  K: 4->4  Acc: 0.100  BT: 0.0000
[Task 6/10]
  Action: SPAWN_NEW  K: 4->5  Acc: 0.830  BT: 0.0050
[Task 7/10]
  Action: SPAWN_NEW  K: 5->6  Acc: 0.875  BT: 0.0042
[Task 8/10]
  Action: SPAWN_NEW  K: 6->7  Acc: 0.815  BT: 0.0036
[Task 9/10]
  Action: SKIP  K: 7->7  Acc: 0.065  BT: 0.0031
[Task 10/10]
  Action: SKIP  K: 7->7  Acc: 0.065  BT: 0.0028

=== SOMA Summary ===
  backward_transfer        : 0.0028
  forward_transfer         : 0.0000
  final_k                  : 7
  spawn_count              : 7
  merge_count              : 0
  tasks_completed          : 10
  targ

In [20]:
# Verify PASS/FAIL
bt = result['backward_transfer']
k = result['final_k']
passed = bt > -0.05 and k < 10
print(f'\nResult: {"PASS" if passed else "FAIL"}')
print(f'BT = {bt:.4f} (target > -0.05)')
print(f'K  = {k} (target < 10)')


Result: PASS
BT = 0.0028 (target > -0.05)
K  = 7 (target < 10)
